In [5]:
from Bio import SeqIO

record = SeqIO.to_dict(SeqIO.parse('../ref_seqs/UP000002473_11622_LASV_Josiah_reference.fasta', 'fasta'))

lasv_gpc = record["sp|P08669|GLYC_LASSJ"].seq


In [6]:
lasv_gpc

Seq('MGQIVTFFQEVPHVIEEVMNIVLIALSVLAVLKGLYNFATCGLVGLVTFLLLCG...WKR')

### first going to do calculations with full region of interest, so AA279-438

In [18]:
sub_record = lasv_gpc[278:438]

In [19]:
#checked, start and end is correct

sub_record 

Seq('CLTRWMLIEAELKCFGNTAVAKCNEKHDEEFCDMLRLFDFNKQAIQRLKAEAQM...FST')

In [30]:
from Bio import SeqIO

def split_fasta(seq, output_file, window=46, overlap=23):

    seq = str(seq)
    seq_len = len(seq)
    step = window-overlap

    record_id = "sp|P08669|GLYC_LASSJ"

    with open(output_file, 'w') as out_f:

        #regular 46aa window
        starts = list(range(0, seq_len - window + 1, step))

        for start in starts:
            subseq = seq[start:start+window]

            out_f.write(
                f">{record_id}_pos_{start + 1}-{start + window}\n" 
            )

            out_f.write(f"{subseq}\n")
        
        # add a C-terminal 46-aa peptide if the last window
        #does not already end at the C-terminus
        last_start = seq_len - window

        if starts and starts[-1] != last_start: #starts[-1] is last element in list starts
            subseq = seq[-window:]

            out_f.write(
                f">{record_id}_pos{last_start+1}-{seq_len}\n"
            )

            out_f.write(f"{subseq}\n")

            
        out_f.write(f">{record_id}_pos_{start+1}-{start+len(subseq)}\n")
        out_f.write(f"{subseq}\n")



In [31]:
split_fasta(seq=sub_record , output_file = 'output.fasta')

In [32]:
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord

AA = "ACDEFGHIKLMNPQRSTVWY"

def dms_scan(input_fasta, output_fasta, include_wt=True):

    records_out = []

    for record in SeqIO.parse(input_fasta, "fasta"):
        seq = str(record.seq)

        # Add WT peptide
        if include_wt:
            records_out.append(
                SeqRecord(
                    Seq(seq),
                    id=f"{record.id}_WT",
                    description=""
                )
            )

        # Single amino-acid substitutions
        for pos, wt_aa in enumerate(seq):

            for mut_aa in AA:

                # Don't generate WT -> WT
                if mut_aa == wt_aa:
                    continue

                mutant_seq = seq[:pos] + mut_aa + seq[pos + 1:]

                records_out.append(
                    SeqRecord(
                        Seq(mutant_seq),
                        id=f"{record.id}_{wt_aa}{pos+1}{mut_aa}",
                        description=""
                    )
                )

    SeqIO.write(records_out, output_fasta, "fasta")

    print(f"Wrote {len(records_out)} sequences to {output_fasta}")

In [33]:
dms_scan(
    input_fasta="output.fasta",
    output_fasta="output_DMS.fasta"
)

Wrote 6125 sequences to output_DMS.fasta


In [12]:
print([i for i in range (0, 5)])

[0, 1, 2, 3, 4]


### do calculations with the entire GP2 region, so AA 260-491

### do calculations only alanine scan